In [39]:
import pandas as pd

In [40]:
df = pd.read_csv("processed_clause_dataset_clean.csv")

print(df.shape)
df.head()

(5694, 9)


,contract_title,clause_type,answer_text,question,context,has_clause,clause_length,clean_text,text_length
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,DISTRIBUTOR AGREEMENT,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,21,distributor agreement,21
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,"7th day of September, 1999.",Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,27,7th day of september 1999,25
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,The term of this Agreement shall be ten (10)...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,273,the term of this agreement shall be ten 10 yea...,176
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,The term of this Agreement shall be ten (10)...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,273,the term of this agreement shall be ten 10 yea...,176
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Renewal Term,If Distributor comp...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,338,if distributor complies with all of the terms ...,220


In [41]:
risk_weights = {

    "Uncapped Liability": 20,

    "Non-Compete": 15,

    "Exclusivity": 10,

    "Anti-Assignment": 5,

    "Termination For Convenience": -10,

    "Governing Law": -5,

    "Insurance": -5
}

In [42]:
print(df.columns)

Index(['contract_title', 'clause_type', 'answer_text', 'question', 'context',
       'has_clause', 'clause_length', 'clean_text', 'text_length'],
      dtype='str')


In [43]:
print(df.columns.tolist())

['contract_title', 'clause_type', 'answer_text', 'question', 'context', 'has_clause', 'clause_length', 'clean_text', 'text_length']


In [44]:
print(df["contract_title"].nunique())

509


In [45]:
df["contract_title"].value_counts().head()

contract_title
ENERGOUSCORP_03_16_2017-EX-10.24-STRATEGIC ALLIANCE AGREEMENT                               28
DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement    27
SoupmanInc_20150814_8-K_EX-10.1_9230148_EX-10.1_Franchise Agreement1                        27
BUFFALOWILDWINGSINC_06_05_1998-EX-10.3-FRANCHISE AGREEMENT                                  25
JOINTCORP_09_19_2014-EX-10.15-FRANCHISE AGREEMENT                                           25
Name: count, dtype: int64

In [46]:
# Creating Risk Weights
risk_weights = {

    # High Risk
    "Uncapped Liability": 25,
    "Non-Compete": 20,
    "Exclusivity": 15,
    "Change Of Control": 10,
    "Anti-Assignment": 10,
    "Minimum Commitment": 10,

    # Protective
    "Termination For Convenience": -10,
    "Insurance": -10,
    "Governing Law": -5
}

In [47]:
#Creating Group Clauses by Contract
contract_clauses = (
    df.groupby("contract_title")["clause_type"]
    .apply(list)
    .reset_index()
)
contract_clauses.head()

,contract_title,clause_type
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,"[Document Name, Expiration Date, Renewal Term,..."
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,"[Parties, Expiration Date, Governing Law, Term..."
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,"[Document Name, Parties, Effective Date, Expir..."
3,ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGR...,"[Document Name, Expiration Date, Governing Law..."
4,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,"[Document Name, Effective Date, Expiration Dat..."


In [48]:
# Function to Calculate Risk Score
def calculate_risk_score(clauses):

    score = 0

    risk_clauses = []

    protective_clauses = []

    for clause in clauses:

        if clause in risk_weights:

            score += risk_weights[clause]

            if risk_weights[clause] > 0:
                risk_clauses.append(clause)

            else:
                protective_clauses.append(clause)

    score = max(0, min(score, 100))

    return score, risk_clauses, protective_clauses

In [49]:
# Applying to all contracts
contract_clauses[
    ["risk_score",
     "risk_clauses",
     "protective_clauses"]
] = contract_clauses["clause_type"].apply(
    lambda x: pd.Series(
        calculate_risk_score(x)
    )
)

In [50]:
# Creating score for risk levels
def risk_level(score):
    if score < 20:
        return "Low Risk"
    elif score < 50:
        return "Medium Risk"
    else:
        return "High Risk"

In [51]:
contract_clauses["risk_level"] = (
    contract_clauses["risk_score"]
    .apply(risk_level)
)

In [53]:
contract_clauses[
    [
        "contract_title",
        "risk_score",
        "risk_level",
        "risk_clauses",
        "protective_clauses"
    ]
].head()

,contract_title,risk_score,risk_level,risk_clauses,protective_clauses
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,60,High Risk,"[Non-Compete, Change Of Control, Anti-Assignme...",[Governing Law]
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,0,Low Risk,[Anti-Assignment],"[Governing Law, Termination For Convenience]"
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,5,Low Risk,[Anti-Assignment],[Governing Law]
3,ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGR...,30,Medium Risk,"[Anti-Assignment, Uncapped Liability]",[Governing Law]
4,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,50,High Risk,"[Non-Compete, Exclusivity, Anti-Assignment, Mi...",[Governing Law]
